# Hardware Test in Pynq-Z2 Board

In [48]:
from pathlib import Path
from typing import Tuple

import numpy as np

from axi_stream_driver import NeuralNetworkOverlay

## Data

In [49]:
test_data = np.load("test_raw.npz", allow_pickle=True)

X_test = test_data["data"]
y_test = test_data["labels"]

X_test = X_test[:128]
y_test = y_test[:128]

print("\nDataset shapes:")
print(f"  Test:  {X_test.shape}, labels: {y_test.shape}")


Dataset shapes:
  Test:  (128, 64), labels: (128,)


### Quantization

This is quantization for 8bits according to axi_stream_driver

In [50]:
# ap_ufixed<7,8>: W=7, I=8, F = W-I = -1
input_fraction_bits = 7 - 8   # = -1
# ap_fixed<14,9>: W=14, I=9, F = W-I = 5
output_fraction_bits = 9 - 14  # = 5

encode_v = np.vectorize(lambda xi: np.uint16(np.round(xi * 2**input_fraction_bits)))
decode_v = np.vectorize(lambda yi: yi * 2**-output_fraction_bits)

# def encode(xi):
#     return np.uint8(np.round(xi * 2**input_fraction_bits))
# def decode(yi):
#     return yi * 2**-output_fraction_bits

# # Input encoding: float → ap_ufixed<7,8> raw bits
# encode_v = np.vectorize(lambda xi: np.uint16(np.round(xi * 2**(7-8))))
# # Output decoding: ap_fixed<14,9> raw bits → float
# decode_v = np.vectorize(lambda yi: yi * 2**(9-14))



## Neural Network

In [51]:
# nn = NeuralNetworkOverlay("myproject.bit", X_test.shape, y_test.shape)

nn = NeuralNetworkOverlay("myproject.bit", X_test.shape, y_test.shape,
    dtype=np.int16,  # or np.int32 for 32-bit output
)

y_hw, latency, throughput = nn.predict(X_test, profile=True, debug=True, encode=encode_v, decode=decode_v)

Transfer OK
Send OK
Receive OK
Classified 128 samples in 0.9968699999999999 seconds (128.40189794055394 inferences / s)


### Evaluate

In [52]:
print(y_hw)

[0.03125 0.      0.40625 0.      0.4375  0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.     ]


## Save

In [53]:
np.savez("y_hw.npz", y_hw)